In [ ]:
import os
# Prevent PyTorch / OpenMP runtime crash on Windows
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import cohere
import speech_recognition as sr
import whisper
from RealtimeTTS import EdgeEngine, TextToAudioStream 

# ==========================================
# 1. INITIALIZE ENGINES & APIS
# ==========================================
COHERE_API_KEY = "cohere_5UxfInxWLUR2jD3wJqONzbGkRyCSTDQSvL0S7WqD43B14D"
co = cohere.Client(COHERE_API_KEY)

print("Loading Whisper STT model (this may take a moment)...")
stt_model = whisper.load_model("base")

recognizer = sr.Recognizer()

print("Initializing EdgeEngine...")
# EdgeEngine is completely stable. We can declare it globally once!
tts_engine = EdgeEngine()
tts_stream = TextToAudioStream(tts_engine)

## ==========================================
# 2. GENERATOR FUNCTION FOR STREAMING
# ==========================================
def stream_cohere_response(prompt):
    """Yields chunks of text from Cohere as they are generated, cleaned for TTS."""
    response = co.chat_stream(
        message=prompt, 
        model="command-r7b-12-2024",
        # This preamble acts as a system prompt to change the AI's behavior
        preamble="You are a conversational voice assistant. Always respond in natural spoken language. Do NOT use markdown formatting, asterisks, or bullet points.",
        connectors=[]
    )

    for event in response:
        if event.event_type == "text-generation":
            # Strip out any asterisks or hashtags that still manage to slip through
            clean_text = event.text.replace("*", "").replace("#", "")
            
            # Print and yield the cleaned text
            print(clean_text, end="", flush=True)
            yield clean_text

# ==========================================
# 3. MAIN CHATBOT LOOP
# ==========================================
def run_chatbot():
    with sr.Microphone() as source:
        print("\nAdjusting for ambient noise... Please wait.")
        recognizer.adjust_for_ambient_noise(source, duration=1.5)
        print("\n--- System Ready! ---")

        while True:
            try:
                print("\nListening (Speak now)...")
                audio_data = recognizer.listen(source)

                print("Transcribing...")
                with open("temp_audio.wav", "wb") as f:
                    f.write(audio_data.get_wav_data())

                # Transcribe using Whisper (fp16=False prevents CPU warning)
                result = stt_model.transcribe("temp_audio.wav", fp16=False)
                user_text = result["text"].strip()

                if not user_text:
                    continue

                print(f"You: {user_text}")

                if user_text.lower() in ["exit", "stop", "quit"]:
                    print("Shutting down chatbot...")
                    break

                print("Bot: ", end="", flush=True)

                # Stream the response natively without crashing
                generator = stream_cohere_response(user_text)
                tts_stream.feed(generator)
                tts_stream.play()
                print()  

            except KeyboardInterrupt:
                print("\nExiting manually...")
                break
            except Exception as e:
                print(f"\nAn error occurred: {e}")
            finally:
                if os.path.exists("temp_audio.wav"):
                    os.remove("temp_audio.wav")

if __name__ == "__main__":
    run_chatbot()

Loading Whisper STT model (this may take a moment)...
Initializing EdgeEngine...

Adjusting for ambient noise... Please wait.

--- System Ready! ---

Listening (Speak now)...
Transcribing...
You: Hello, good morning.
Bot: Hello! How can I help you today?

Listening (Speak now)...
Transcribing...

Listening (Speak now)...
Transcribing...
You: with my computer.
Bot: I'm sorry, I'm not sure I understand your request. Could you please provide more details or clarify what you need help with? I'm here to assist you with any questions or tasks you might have.

Listening (Speak now)...
Transcribing...
You: I have a problem with my computer, can you help me?
Bot: I'd be happy to help! Please provide more details about the issue you're experiencing with your computer. What specific problems are you encountering? When did the issue start? Have you tried any troubleshooting steps already? The more information you can give me, the better I can assist you in resolving the problem.

Listening (Speak 